# FingerBall v2 — Stage 2C：手動ROI・手部マスク・21点骨格動画
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_v2_Stage2C_ManualROI_HandMask_Landmarks.ipynb)

**今回の短期目標**：リリース前後の指定フレーム区間だけ、手領域のマスクと21点2D骨格を重畳したMP4、ランドマーク座標CSV、ROI/検出品質CSV、2値マスクPNGを出力します。

- フレーム区間・ROIの3キーフレームは **Mac対応ipywidgetsスライダー**で指定。間のROIは線形補間します。
- 骨格は**MediaPipe Hand Landmarker（21点2D）**。WiLoRの未検証2Dカメラ投影を使わず、手の見え方を優先します。
- マスクは**ROIと21点から初期化したGrabCut暫定マスク**で、教師ありhand segmentationではありません。指やボールの正確な境界を保証しません。
- 検出できなかったフレームは **NaN**、マスクは空として残します。存在しない骨格を捏造しません。
- 研究データ・画像・重みはGitHubに入れないでください。Colabによるデータ処理について研究倫理と同意条件を確認してください。

## 0. セットアップとGoogle Drive

In [ ]:
%pip -q install "mediapipe==0.10.35" "opencv-python-headless>=4.8" ipywidgets pandas matplotlib tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 設定 — 基本的にここだけ変更

In [ ]:
from pathlib import Path
from google.colab import files, output
from IPython.display import display
import numpy as np, pandas as pd, cv2, mediapipe as mp
import ipywidgets as widgets
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import urllib.request, shutil, json, zipfile
from datetime import datetime
output.enable_custom_widget_manager()

PROJECT_PATH = ''  # 例 '/content/drive/MyDrive/母指回内トラッキング'
VIDEO_PATH = ''    # 空なら手動アップロード
START_FRAME, END_FRAME = 175, 210
ANCHOR_FRAMES = (175, 190, 205)  # 3つのROIキーフレーム（解析区間内）
CROP_SCALE = 2.0     # 手のBOXを拡大してモデルに入力
HAND_MIN_DET = 0.15
HAND_MIN_PRESENCE = 0.2
HAND_MIN_TRACK = 0.2
RUN_GRABCUT = True
RELEASE_ZOOM = 3.0
OUTPUT_DIR_NAME = 'FingerBall_v2_stage2c_results'
TASK_URLS = [
 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task',
 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task']
def find_project():
    if PROJECT_PATH:
        p=Path(PROJECT_PATH)
        if not p.is_dir():raise FileNotFoundError(p)
        return p
    roots=[Path('/content/drive/MyDrive'),Path('/content/drive/Shareddrives')]
    for root in roots:
        if (root/'母指回内トラッキング').is_dir():return root/'母指回内トラッキング'
        if root.name=='Shareddrives' and root.is_dir():
            for d in root.iterdir():
                if (d/'母指回内トラッキング').is_dir():return d/'母指回内トラッキング'
    raise FileNotFoundError('PROJECT_PATHを指定してください')
project=find_project()
if not VIDEO_PATH:
    uploaded=files.upload()
    if not uploaded:raise RuntimeError('動画が指定されません')
    VIDEO_PATH='/content/'+next(iter(uploaded))
video_original=Path(VIDEO_PATH)
if not video_original.is_file():raise FileNotFoundError(video_original)
cache=Path('/content/fingerball_stage2c');cache.mkdir(exist_ok=True)
if str(video_original).startswith('/content/drive/'):
    video=cache/('input_'+video_original.name)
    if not video.exists() or video.stat().st_size!=video_original.stat().st_size:shutil.copy2(video_original,video)
else:video=video_original
cap=cv2.VideoCapture(str(video))
if not cap.isOpened():raise RuntimeError('動画が開けません')
TOTAL=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FPS=float(cap.get(cv2.CAP_PROP_FPS))
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
assert 0<=START_FRAME<=END_FRAME<TOTAL
assert len(ANCHOR_FRAMES)>=2 and min(ANCHOR_FRAMES)>=START_FRAME and max(ANCHOR_FRAMES)<=END_FRAME
assert sorted(set(ANCHOR_FRAMES))==list(ANCHOR_FRAMES)
print('video:',video_original.name,'total:',TOTAL,'fps:',FPS,'size:',W,H,'range:',START_FRAME,END_FRAME)

## 2. キーフレームのROIをスライダーで指定

**必ずこのセルの実行後、表示されたフレームを順に選んでBOXを確定してください。** `ANCHOR_FRAMES` で例えば 175,190,205 の3箇所を手動設定します。フレーム間はROIの位置と大きさを線形補間します。後続セルはすべてのBOXが確定しているかチェックします。

In [ ]:
anchor_boxes={}
def frame_at(index):
    cap=cv2.VideoCapture(str(video));cap.set(cv2.CAP_PROP_POS_FRAMES,int(index))
    ok,im=cap.read();cap.release()
    if not ok:raise RuntimeError('Read frame '+str(index))
    return im
frames_cache={i:frame_at(i) for i in ANCHOR_FRAMES}
chooser=widgets.Dropdown(options=list(ANCHOR_FRAMES),description='フレーム')
xr=widgets.IntRangeSlider(value=(int(W*.35),int(W*.65)),min=0,max=W-1,description='X範囲',continuous_update=False,layout=widgets.Layout(width='95%'))
yr=widgets.IntRangeSlider(value=(int(H*.2),int(H*.5)),min=0,max=H-1,description='Y範囲',continuous_update=False,layout=widgets.Layout(width='95%'))
preview=widgets.Image(format='png')
btn=widgets.Button(description='このフレームのBOXを確定',button_style='success',icon='check')
msg=widgets.HTML()
def roi_from_sliders():
    x0,x1=xr.value;y0,y1=yr.value
    if x1-x0<12 or y1-y0<12:return None
    return np.array([x0,y0,x1,y1],dtype=float)
def redraw(*args):
    img=frames_cache[chooser.value].copy()
    b=roi_from_sliders()
    if b is not None:
        x0,y0,x1,y1=map(int,b)
        cv2.rectangle(img,(x0,y0),(x1,y1),(0,255,0),3)
    scale=min(1,960/W)
    if scale<1:img=cv2.resize(img,None,fx=scale,fy=scale)
    _,buf=cv2.imencode('.png',img)
    preview.value=buf.tobytes()
    msg.value='確定済み: '+', '.join(map(str,sorted(anchor_boxes)))
def choose(change):
    if change['new'] in anchor_boxes:
        b=anchor_boxes[change['new']]
        xr.value=(int(b[0]),int(b[2]));yr.value=(int(b[1]),int(b[3]))
    redraw()
def confirm(button):
    b=roi_from_sliders()
    if b is None:
        msg.value='<b>BOXの幅と高さは12px以上必要です</b>'
        return
    anchor_boxes[int(chooser.value)]=b.copy()
    redraw()
    print('確定:',chooser.value,b.astype(int).tolist())
chooser.observe(choose,names='value')
xr.observe(redraw,names='value')
yr.observe(redraw,names='value')
btn.on_click(confirm)
redraw()
display(widgets.VBox([chooser,preview,xr,yr,btn,msg]))
print('すべてのキーフレームでBOXを確定してから次のセルへ進んでください')

## 3. ROIの補間プレビュー・MediaPipeモデルの取得

In [ ]:
if sorted(anchor_boxes)!=list(ANCHOR_FRAMES):
    raise RuntimeError('未設定のROIキーフレーム: '+str(sorted(set(ANCHOR_FRAMES)-set(anchor_boxes))))
keys=np.array(sorted(anchor_boxes),dtype=int)
def interp_box(i):
    if i<=keys[0]:return anchor_boxes[int(keys[0])].copy()
    if i>=keys[-1]:return anchor_boxes[int(keys[-1])].copy()
    k=np.searchsorted(keys,i);a,b=int(keys[k-1]),int(keys[k])
    t=(i-a)/(b-a)
    return (1-t)*anchor_boxes[a]+t*anchor_boxes[b]
fig,axs=plt.subplots(1,len(ANCHOR_FRAMES),figsize=(15,5))
for ax,i in zip(np.ravel(axs),ANCHOR_FRAMES):
    img=frames_cache[i].copy();q=np.rint(interp_box(i)).astype(int)
    cv2.rectangle(img,tuple(q[:2]),tuple(q[2:]),(0,255,0),3)
    ax.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB));ax.set_title(f'Frame {i}');ax.axis('off')
plt.show()
task_path=cache/'hand_landmarker.task'
if task_path.exists() and task_path.stat().st_size<1000000:task_path.unlink()
if not task_path.exists():
    errors=[]
    for url in TASK_URLS:
        tmp=cache/'hand_download.tmp'
        try:
            req=urllib.request.Request(url,headers={'User-Agent':'Mozilla/5.0'})
            with urllib.request.urlopen(req,timeout=90) as response,open(tmp,'wb') as out:shutil.copyfileobj(response,out)
            if tmp.stat().st_size<1000000:raise RuntimeError('Model file too small')
            tmp.replace(task_path);break
        except Exception as e:
            errors.append(str(e));tmp.unlink(missing_ok=True)
    if not task_path.exists():raise RuntimeError('モデル取得失敗: '+repr(errors))
print('HandLandmarker model:',task_path.stat().st_size,'bytes')

## 4. 指定区間の全フレームで21点を検出、マスクを推定しCSV化

- Hand Landmarkerは指定ROIを拡大した切り出し画像へ適用し、**元動画のピクセル座標へ逆変換**します。
- フレームごとに21行をCSVへ出力し、失敗した点の x/y/z はNaNです。
- マスクはGrabCutからの**形状近似**です。ボール・前腕を含む可能性があり、精密なhand segmentationとは区別します。
- このNotebookは実験的な診断用実装であり、WiLoR公式投影は使用していません。

In [ ]:
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
LANDMARK_NAMES=['WRIST','THUMB_CMC','THUMB_MCP','THUMB_IP','THUMB_TIP',
                'INDEX_MCP','INDEX_PIP','INDEX_DIP','INDEX_TIP',
                'MIDDLE_MCP','MIDDLE_PIP','MIDDLE_DIP','MIDDLE_TIP',
                'RING_MCP','RING_PIP','RING_DIP','RING_TIP',
                'PINKY_MCP','PINKY_PIP','PINKY_DIP','PINKY_TIP']
EDGES=[(0,1),(1,2),(2,3),(3,4),
       (0,5),(5,6),(6,7),(7,8),
       (0,9),(9,10),(10,11),(11,12),
       (0,13),(13,14),(14,15),(15,16),
       (0,17),(17,18),(18,19),(19,20)]
def roi_crop(frame,box,size=512):
    cx=(box[0]+box[2])/2;cy=(box[1]+box[3])/2
    side=max(box[2]-box[0],box[3]-box[1])*CROP_SCALE
    M=np.float32([[size/side,0,size/2-cx*size/side],
                  [0,size/side,size/2-cy*size/side]])
    patch=cv2.warpAffine(frame,M,(size,size),flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_CONSTANT,borderValue=(114,114,114))
    return patch,np.array([cx,cy],float),float(side)
def make_mask(patch,uv,grabcut=True):
    size=patch.shape[0]
    pts=np.rint(uv).astype(np.int32)
    if not np.isfinite(uv).all():return np.zeros((size,size),np.uint8),'none'
    hull=cv2.convexHull(pts)
    raw=np.zeros((size,size),np.uint8)
    cv2.fillConvexPoly(raw,hull,255)
    # A soft region wider than the skeletal hull approximates hand thickness
    spread=max(9,int(size*.035));kern=np.ones((spread,spread),np.uint8)
    probable=cv2.dilate(raw,kern,iterations=2)
    if not grabcut:return probable,'dilated_hull'
    gc=np.full((size,size),cv2.GC_BGD,dtype=np.uint8)
    gc[probable>0]=cv2.GC_PR_FGD
    gc[raw>0]=cv2.GC_FGD
    # Ensure at least some definite foreground and background for GrabCut
    center=cv2.erode(raw,np.ones((7,7),np.uint8),iterations=1)
    gc[center>0]=cv2.GC_FGD
    if (gc==cv2.GC_FGD).sum()<30 or (gc==cv2.GC_BGD).sum()<30:
        return probable,'dilated_hull_fallback'
    try:
        cv2.grabCut(patch,gc,None,np.zeros((1,65),np.float64),
                    np.zeros((1,65),np.float64),3,cv2.GC_INIT_WITH_MASK)
        mask=np.isin(gc,[cv2.GC_FGD,cv2.GC_PR_FGD]).astype(np.uint8)*255
        if mask.sum()<1000:return probable,'dilated_hull_fallback'
        return mask,'grabcut_provisional'
    except cv2.error:
        return probable,'dilated_hull_fallback'
def mask_to_frame(mask,center,side):
    m=np.float32([[512/side,0,256-center[0]*512/side],
                  [0,512/side,256-center[1]*512/side]])
    inv=cv2.invertAffineTransform(m)
    return cv2.warpAffine(mask,inv,(W,H),flags=cv2.INTER_NEAREST,
                          borderMode=cv2.BORDER_CONSTANT,borderValue=0)
def points_to_original(uv,center,side):
    return center[None,:]+(uv-256.)*side/512.
settings=mp_vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=str(task_path)),
    running_mode=mp_vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=HAND_MIN_DET,
    min_hand_presence_confidence=HAND_MIN_PRESENCE,
    min_tracking_confidence=HAND_MIN_TRACK)
out_dir=project/OUTPUT_DIR_NAME/video_original.stem
out_dir.mkdir(parents=True,exist_ok=True)
masks_dir=out_dir/'masks_png';masks_dir.mkdir(exist_ok=True)
landmark_rows=[];qc_rows=[];results={}
cap=cv2.VideoCapture(str(video))
if not cap.isOpened():raise RuntimeError('Cannot open video')
last_ms=-1
with mp_vision.HandLandmarker.create_from_options(settings) as detector:
    for f in tqdm(range(START_FRAME,END_FRAME+1),desc='Release hand analysis'):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(f))
        ok,frame=cap.read()
        if not ok:raise RuntimeError('Unreadable frame: '+str(f))
        bbox=interp_box(f)
        patch,center,side=roi_crop(frame,bbox)
        rgb=cv2.cvtColor(patch,cv2.COLOR_BGR2RGB)
        timestamp=max(last_ms+1,int(round(1000*f/FPS)) if FPS>0 else f*33)
        last_ms=timestamp
        pred=detector.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB,
                                                data=np.ascontiguousarray(rgb)),timestamp)
        best=None
        for k,lm in enumerate(pred.hand_landmarks):
            uv=np.array([[q.x*512,q.y*512] for q in lm],dtype=float)
            # Prefer the hand nearest the manually defined ROI center
            dist=float(np.linalg.norm(np.median(uv,axis=0)-np.array([256,256]))/512)
            if best is None or dist<best[0]:best=(dist,k,uv,lm)
        selected=False;score=np.nan;handedness=''
        mask_full=np.zeros((H,W),np.uint8);mask_method='none'
        xy=np.full((21,2),np.nan);z=np.full(21,np.nan)
        if best is not None:
            dist,k,uv,lm=best
            original=points_to_original(uv,center,side)
            valid=(np.isfinite(original).all(axis=1) & (original[:,0]>=0)
                   & (original[:,0]<W) & (original[:,1]>=0) & (original[:,1]<H))
            # Reject hands largely outside the ROI crop; never reuse old landmark guesses
            if valid.mean()>=.75 and dist<.46:
                selected=True
                xy=original
                z=np.asarray([p.z for p in lm],dtype=float)
                if pred.handedness and k<len(pred.handedness) and pred.handedness[k]:
                    handedness=pred.handedness[k][0].category_name
                    score=float(pred.handedness[k][0].score)
                m_crop,mask_method=make_mask(patch,uv,RUN_GRABCUT)
                mask_full=mask_to_frame(m_crop,center,side)
        name=f'frame_{f:06d}.png'
        cv2.imwrite(str(masks_dir/name),mask_full)
        records={'frame':f,'time_s':f/FPS if FPS>0 else np.nan,'hand_detected':int(selected),
                 'roi_source':'manual_keyframe' if f in anchor_boxes else 'linear_interpolation',
                 'mask_method':mask_method,'mask_area_px':int(np.count_nonzero(mask_full)),
                 'handedness_reported':handedness,'handedness_score':score,
                 'roi_x0':bbox[0],'roi_y0':bbox[1],'roi_x1':bbox[2],'roi_y1':bbox[3]}
        qc_rows.append(records)
        for j in range(21):
            landmark_rows.append({'frame':f,'time_s':records['time_s'],'landmark_id':j,
                                  'landmark_name':LANDMARK_NAMES[j],
                                  'x_px':xy[j,0],'y_px':xy[j,1],
                                  'x_norm':xy[j,0]/W,'y_norm':xy[j,1]/H,
                                  'z_relative':z[j],'detected':int(selected),
                                  'roi_source':records['roi_source'],
                                  'handedness_reported':handedness})
        results[f]={'xy':xy,'mask':mask_full,'roi':bbox,'detected':selected,'mask_method':mask_method}
cap.release()
landmarks_csv=out_dir/'hand_21_landmarks_long.csv'
qc_csv=out_dir/'tracking_mask_qc.csv'
pd.DataFrame(landmark_rows).to_csv(landmarks_csv,index=False)
qdf=pd.DataFrame(qc_rows);qdf.to_csv(qc_csv,index=False)
print('Detected:',int(qdf.hand_detected.sum()),'/',len(qdf))
print('Saved coordinates:',landmarks_csv)
print('Mask methods:',qdf.mask_method.value_counts().to_dict())

## 5. マスク＋21点骨格を重ねたMP4・拡大動画を生成

- 緑枠：手動アンカー/補間ROI
- 緑の半透明領域：GrabCutまたは骨格凸包の**暫定的な手領域**
- 白線と赤点：MediaPipeの21点2D骨格（検出成功フレームのみ）
- 未検出フレーム：マスクも骨格も描画せず、QCラベルを表示
- 解析元フレーム番号と手検出状態を明記します。

In [ ]:
def annotated_frame(img,f):
    result=results[f]
    a=img.copy();b=np.rint(result['roi']).astype(int)
    cv2.rectangle(a,tuple(b[:2]),tuple(b[2:]),(0,230,0),2)
    mask=result['mask']
    if np.any(mask):
        tint=a.copy();tint[mask>0]=(20,175,20)
        a=cv2.addWeighted(tint,.30,a,.70,0)
        contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(a,contours,-1,(0,255,0),2)
    if result['detected']:
        xy=np.rint(result['xy']).astype(int)
        for i,j in EDGES:
            cv2.line(a,tuple(xy[i]),tuple(xy[j]),(255,255,255),2)
        for i,(x,y) in enumerate(xy):
            cv2.circle(a,(int(x),int(y)),3,(30,30,255),-1)
    label=f"Frame {f} | hand={'YES' if result['detected'] else 'NO'} | {result['mask_method']}"
    cv2.rectangle(a,(0,0),(min(W,760),39),(0,0,0),-1)
    cv2.putText(a,label,(12,27),cv2.FONT_HERSHEY_SIMPLEX,.65,(255,255,255),2)
    return a
overlay_path=out_dir/'hand_mask_skeleton_overlay.mp4'
zoom_path=out_dir/f'release_zoom_{START_FRAME}_{END_FRAME}.mp4'
def writer_or_fail(path,dimensions):
    writer=cv2.VideoWriter(str(path),cv2.VideoWriter_fourcc(*'mp4v'),
                           FPS if FPS>0 else 30,dimensions)
    if not writer.isOpened():raise RuntimeError('Cannot open writer: '+str(path))
    return writer
full_writer=writer_or_fail(overlay_path,(W,H))
zoom_writer=writer_or_fail(zoom_path,(768,768))
cap=cv2.VideoCapture(str(video))
try:
    for f in tqdm(range(START_FRAME,END_FRAME+1),desc='Render overlays'):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(f))
        ok,img=cap.read()
        if not ok:raise RuntimeError('Frame read failed '+str(f))
        overlay=annotated_frame(img,f)
        full_writer.write(overlay)
        b=results[f]['roi']
        cx=(b[0]+b[2])/2;cy=(b[1]+b[3])/2
        side=max(b[2]-b[0],b[3]-b[1])*RELEASE_ZOOM
        affine=np.float32([[768/side,0,384-cx*768/side],[0,768/side,384-cy*768/side]])
        zoom=cv2.warpAffine(overlay,affine,(768,768),borderMode=cv2.BORDER_CONSTANT,
                            borderValue=(25,25,25))
        cv2.putText(zoom,f'Frame {f}',(12,740),cv2.FONT_HERSHEY_SIMPLEX,.8,(255,255,255),2)
        zoom_writer.write(zoom)
finally:
    cap.release();full_writer.release();zoom_writer.release()
print('Overlay:',overlay_path)
print('Zoom:',zoom_path)
fig,axs=plt.subplots(1,3,figsize=(15,4))
for ax,f in zip(axs,np.linspace(START_FRAME,END_FRAME,3).astype(int)):
    ax.imshow(cv2.cvtColor(annotated_frame(frame_at(int(f)),int(f)),cv2.COLOR_BGR2RGB))
    ax.set_title(f'Frame {f}');ax.axis('off')
plt.show()

## 6. マスクZIP・実行レポート保存とQC

In [ ]:
masks_zip=out_dir/'hand_masks_png.zip'
with zipfile.ZipFile(masks_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for mask_file in sorted(masks_dir.glob('*.png')):
        z.write(mask_file,arcname=mask_file.name)
report={
 'source_video':video_original.name,
 'created_at':datetime.now().isoformat(timespec='seconds'),
 'video_dimensions':[W,H],
 'fps':FPS,
 'frame_range_inclusive':[START_FRAME,END_FRAME],
 'n_frames':END_FRAME-START_FRAME+1,
 'anchor_frames':list(ANCHOR_FRAMES),
 'anchor_boxes':{str(k):v.tolist() for k,v in anchor_boxes.items()},
 'hand_detected_frames':int(qdf.hand_detected.sum()),
 'hand_detection_fraction':float(qdf.hand_detected.mean()),
 'mask_methods':{str(k):int(v) for k,v in qdf.mask_method.value_counts().items()},
 'outputs':{'landmarks_csv':str(landmarks_csv),'qc_csv':str(qc_csv),
            'overlay_mp4':str(overlay_path),'zoom_mp4':str(zoom_path),
            'masks_zip':str(masks_zip)},
 'model':'MediaPipe Hand Landmarker float16',
 'important_limitations':[
  'The hand mask is an approximate GrabCut/landmark-hull segmentation, not validated semantic segmentation',
  'Landmark x/y are MediaPipe 2D points mapped to original frame, not WiLoR 3D projection',
  'ROI is linearly interpolated between manually chosen boxes, not automatically ground-truth tracked',
  'Missing detections are NaN; no fabricated points or mask',
  'No ball detection or release-frame classification is implemented']
}
report_path=out_dir/'stage2c_report.json'
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf8')
print(json.dumps(report,ensure_ascii=False,indent=2))
print('Result folder:',out_dir)
plt.figure(figsize=(12,3))
plt.plot(qdf.frame,qdf.hand_detected,drawstyle='steps-mid',label='Detected')
plt.xlabel('Frame');plt.ylabel('Hand detected');plt.ylim(-.1,1.1);plt.grid(True);plt.legend();plt.show()

## 評価基準と今後のWiLoR移植

**今回の達成条件**：175〜210などの指定区間で、画像に重なる**手マスク・21点骨格のMP4**、各フレーム×21点の**元画像座標CSV**、欠損と処理方法のQCが生成されること。

**評価すべき点**：(1) ROIが全対象フレームで投球手を含むか、(2) MediaPipeの21点が実際の指上に重なるか、(3) GrabCutマスクがボール・前腕・背景を過剰に含まないか。正しい手指の検出率は目視またはGT付与で評価してください。

**次の発展**：既存WiLoRプロジェクトの正規パッチ前処理、MANO/camera-to-image projectionを移植し、MediaPipe 2DとWiLoR 2Dを同じ座標系で比較。その後に示指・中指関節角度と指球相互作用へ進みます。